# F1 Pit Stop — v12 Pipeline + 4-Way Blend (Single Notebook)
### LGB(Optuna) · XGB(Optuna) · CatBoost · RealMLP-A · RealMLP-B · Stacking → 4-way Blend
**New vs v9:** Field-level competitor features (avg/max tyre age, pit rate, compound splits) ·
Safety car proxy detection (laps_since_sc, in_sc_window) ·
LGB Optuna hyperparameter tuning (like XGB)
**Runtime:** ~50 min on T4x2

In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
import torch
print(f"CUDA: {torch.cuda.is_available()} | GPUs: {torch.cuda.device_count()}")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
%%time
import subprocess
for pkg in ["pytabkit", "lightgbm>=4.3.0", "xgboost>=2.0.0",
            "colorama", "optuna"]:
    subprocess.run(["pip", "install", "-q", pkg], check=True)
print("Done.")


In [ ]:
import warnings; warnings.filterwarnings('ignore')
import random, gc, json, glob, os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from colorama import Fore, Style
from importlib.metadata import version
from scipy.stats import rankdata
from scipy.optimize import minimize

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import KBinsDiscretizer, TargetEncoder

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import lightgbm as lgb
import xgboost as xgb
from pytabkit import RealMLP_TD_Classifier

TRAIN_PATH  = "/kaggle/input/competitions/playground-series-s6e5/train.csv"
TEST_PATH   = "/kaggle/input/competitions/playground-series-s6e5/test.csv"
SUB_PATH    = "/kaggle/input/competitions/playground-series-s6e5/sample_submission.csv"
ORIG_PATH   = "/kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv"
# Ergast F1 dataset — adjust slug to match your attached Kaggle dataset
ERGAST_BASE       = "/kaggle/input/datasets/cjgdev/formula-1-race-data-19502017/"
ERGAST_RACES      = ERGAST_BASE + "races.csv"
ERGAST_DRIVERS    = ERGAST_BASE + "drivers.csv"
ERGAST_RESULTS    = ERGAST_BASE + "results.csv"
ERGAST_QUAL       = ERGAST_BASE + "qualifying.csv"
ERGAST_PITSTOPS   = ERGAST_BASE + "pitStops.csv"
ERGAST_LAP_TIMES  = ERGAST_BASE + "lapTimes.csv"
ERGAST_DRV_STAND  = ERGAST_BASE + "driverStandings.csv"
ERGAST_CONS_STAND = ERGAST_BASE + "constructorStandings.csv"
ERGAST_CONSTRUCTORS = ERGAST_BASE + "constructors.csv"
OUT         = "/kaggle/working/"
SEED, FOLDS = 42, 5

# Public notebook output roots — used for the blend stage at the end
# Kept only the 3 most diverse sources; near-duplicates removed
NB_ROOTS = [
    "/kaggle/input/notebooks/mohamedsadokaissaoui/ps6e5-ensemble-0-95314-best-score",
    "/kaggle/input/notebooks/yekenot/ps-s6-e5-realmlp-pytabkit",
    "/kaggle/input/notebooks/analyticaobscura/pit-or-stay-f1-strategy-1",
]

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
seed_all(SEED)
print(f"PyTabKit: {version('pytabkit')} | LGB: {lgb.__version__} | XGB: {xgb.__version__}")


In [ ]:
%%time
train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
sub   = pd.read_csv(SUB_PATH)

try:
    orig = pd.read_csv(ORIG_PATH)
    if 'Normalized_TyreLife' in orig.columns:
        orig = orig.drop('Normalized_TyreLife', axis=1)
    HAS_ORIG = True
    print(f"Original dataset: {orig.shape}")
except Exception as e:
    orig = None; HAS_ORIG = False
    print(f"No original dataset ({e})")

TEST_IDS = test['id'].values.copy()
print(f"Train: {train.shape} | Test: {test.shape}")
print("Target dist:", train['PitNextLap'].value_counts(normalize=True).round(3).to_dict())


In [ ]:
%%time
import unicodedata

def norm_driver(s):
    s = str(s).strip().split()[-1].lower()
    return ''.join(c for c in unicodedata.normalize('NFD', s)
                   if unicodedata.category(c) != 'Mn')

def norm_circuit(s):
    s = str(s).strip().lower()
    s = s.replace('grand prix', '').replace(' gp', '').strip()
    return s

# Backward-compat aliases used elsewhere in the notebook
_ndrv  = norm_driver
_nrace = norm_circuit

def _try_load(base, *filenames):
    for fn in filenames:
        for enc in ('utf-8', 'latin-1'):
            try:
                df = pd.read_csv(base + fn, encoding=enc)
                print(f"  ✅ {fn}: {len(df):,} rows")
                return df
            except UnicodeDecodeError:
                continue
            except Exception:
                break
    print(f"  ❌ not found: {filenames}")
    return None

print("Loading Ergast F1 dataset...")
HAS_ERGAST = False

# Initialise all lookup tables (populated below if data loads)
grid_lookup       = pd.DataFrame(columns=['_dk','_rk','_yr','grid','constructorId'])
quali_lookup      = pd.DataFrame(columns=['_dk','_rk','_yr','quali_pos'])
ckt_pit_stats     = pd.DataFrame(columns=['_rk','ckt_pit_avg','ckt_pit_std','ckt_pit_n'])
drv_pit_stats     = pd.DataFrame(columns=['_dk','drv_pit_avg','drv_pit_std'])
ckt_deg_rates     = {}
drv_season_stand  = pd.DataFrame(columns=['_dk','_yr','drv_season_pts','drv_season_pos'])
cons_season_stand = pd.DataFrame(columns=['_tk','_yr','cons_season_pts','cons_season_pos'])
drv_team_yr       = pd.DataFrame(columns=['_dk','_yr','_cid','_tk'])

try:
    e_races  = _try_load(ERGAST_BASE, "races.csv")
    e_drivers= _try_load(ERGAST_BASE, "drivers.csv")
    e_results= _try_load(ERGAST_BASE, "results.csv")
    e_qual   = _try_load(ERGAST_BASE, "qualifying.csv", "qualifyings.csv")
    e_pits   = _try_load(ERGAST_BASE, "pitStops.csv", "pit_stops.csv")
    e_laps   = _try_load(ERGAST_BASE, "lapTimes.csv", "lap_times.csv")
    e_dstand = _try_load(ERGAST_BASE, "driverStandings.csv", "driver_standings.csv")
    e_cstand = _try_load(ERGAST_BASE, "constructorStandings.csv", "constructor_standings.csv")
    e_cons   = _try_load(ERGAST_BASE, "constructors.csv")
    _required = [e_races, e_drivers, e_results, e_qual, e_pits, e_dstand, e_cstand, e_cons]
    if all(x is not None for x in _required):
        HAS_ERGAST = True
    else:
        print("  ⚠️  Some Ergast files missing — HAS_ERGAST=False")
except Exception as _ex:
    print(f"  Ergast load error: {_ex}")

if HAS_ERGAST:
    e_races['_rk'] = e_races['name'].map(norm_circuit)
    e_races['_yr'] = e_races['year'].astype(int)
    race_lookup = e_races[['raceId','_rk','_yr','circuitId']].copy()

    e_drivers['_dk'] = e_drivers['surname'].map(norm_driver)
    drv_lookup = e_drivers[['driverId','_dk']].drop_duplicates('driverId')

    # Results → grid position + constructorId per driver/race/year
    e_results_m = e_results.merge(drv_lookup, on='driverId', how='left')
    e_results_m = e_results_m.merge(race_lookup[['raceId','_rk','_yr']], on='raceId', how='left')
    e_results_m['grid'] = pd.to_numeric(e_results_m['grid'], errors='coerce')
    grid_lookup = (e_results_m[['_dk','_rk','_yr','grid','constructorId']]
                   .dropna(subset=['grid']).copy())

    # Qualifying → position per driver/race/year
    e_qual_m = e_qual.merge(drv_lookup, on='driverId', how='left')
    e_qual_m = e_qual_m.merge(race_lookup[['raceId','_rk','_yr']], on='raceId', how='left')
    e_qual_m['position'] = pd.to_numeric(e_qual_m['position'], errors='coerce')
    quali_lookup = (e_qual_m[['_dk','_rk','_yr','position']]
                    .rename(columns={'position':'quali_pos'})
                    .dropna(subset=['quali_pos']).copy())

    # Pit stops → circuit and driver historical pit lap priors
    e_pits_m = e_pits.merge(drv_lookup, on='driverId', how='left')
    e_pits_m = e_pits_m.merge(race_lookup[['raceId','_rk','_yr']], on='raceId', how='left')
    e_pits_m['lap'] = pd.to_numeric(e_pits_m['lap'], errors='coerce')

    ckt_pit_stats = (e_pits_m.dropna(subset=['lap'])
                     .groupby('_rk')['lap']
                     .agg(ckt_pit_avg='mean', ckt_pit_std='std', ckt_pit_n='count')
                     .reset_index())
    ckt_pit_stats['ckt_pit_std'] = ckt_pit_stats['ckt_pit_std'].fillna(8.0)
    ckt_pit_stats = ckt_pit_stats[ckt_pit_stats['ckt_pit_n'] >= 10].copy()

    drv_pit_stats = (e_pits_m.dropna(subset=['lap'])
                     .groupby('_dk')['lap']
                     .agg(drv_pit_avg='mean', drv_pit_std='std')
                     .reset_index())
    drv_pit_stats['drv_pit_std'] = drv_pit_stats['drv_pit_std'].fillna(7.0)
    print(f"  Circuit pit priors: {len(ckt_pit_stats)} circuits | Driver pit priors: {len(drv_pit_stats)} drivers")

    # Lap times → circuit degradation rate (ms per lap from linear fit)
    if e_laps is not None:
        e_laps_m = (e_laps
                    .merge(race_lookup[['raceId','_rk','_yr']], on='raceId', how='left')
                    .dropna(subset=['_rk']))
        e_laps_m['milliseconds'] = pd.to_numeric(e_laps_m['milliseconds'], errors='coerce')
        e_laps_m = e_laps_m.dropna(subset=['milliseconds']).sort_values(['raceId','driverId','lap'])
        from scipy.stats import linregress
        ckt_deg_rates = {}
        for rk, grp in e_laps_m.dropna(subset=['lap','milliseconds']).groupby('_rk'):
            if len(grp) < 50:
                continue
            try:
                slope, _, r, _, _ = linregress(grp['lap'].values, grp['milliseconds'].values)
                ckt_deg_rates[rk] = {'ckt_deg_rate_ms_per_lap': slope, 'ckt_deg_r2': r**2}
            except:
                pass
        print(f"  Circuit degradation rates: {len(ckt_deg_rates)} circuits")
    else:
        print("  ⚠️  lapTimes not loaded — skipping circuit degradation rates")

    # Constructor name key lookup
    e_cons_nm = e_cons.copy()
    e_cons_nm['_tk'] = e_cons_nm['name'].map(norm_driver)
    cons_name_lookup = e_cons_nm[['constructorId','_tk']].drop_duplicates('constructorId')

    # Driver standings → end-of-season position per (driver, year)
    e_dstand_m = (e_dstand
                  .merge(drv_lookup, on='driverId', how='left')
                  .merge(race_lookup[['raceId','_yr']], on='raceId', how='left')
                  .dropna(subset=['_dk','_yr']))
    e_dstand_m['points']   = pd.to_numeric(e_dstand_m['points'],   errors='coerce').fillna(0)
    e_dstand_m['position'] = pd.to_numeric(e_dstand_m['position'], errors='coerce').fillna(20)
    drv_season_stand = (e_dstand_m
                        .sort_values('raceId').groupby(['_dk','_yr']).last().reset_index()
                        [['_dk','_yr','points','position']]
                        .rename(columns={'points':'drv_season_pts','position':'drv_season_pos'}))

    # Constructor standings → end-of-season position per (team, year)
    e_cstand_m = (e_cstand
                  .merge(cons_name_lookup, on='constructorId', how='left')
                  .merge(race_lookup[['raceId','_yr']], on='raceId', how='left')
                  .dropna(subset=['_tk','_yr']))
    e_cstand_m['points']   = pd.to_numeric(e_cstand_m['points'],   errors='coerce').fillna(0)
    e_cstand_m['position'] = pd.to_numeric(e_cstand_m['position'], errors='coerce').fillna(10)
    cons_season_stand = (e_cstand_m
                         .sort_values('raceId').groupby(['_tk','_yr']).last().reset_index()
                         [['_tk','_yr','points','position']]
                         .rename(columns={'points':'cons_season_pts','position':'cons_season_pos'}))

    # Driver → constructor team per year (from results)
    drv_team_yr = (grid_lookup
                   .groupby(['_dk','_yr'])['constructorId']
                   .agg(lambda x: x.mode()[0] if len(x) else -1)
                   .reset_index().rename(columns={'constructorId':'_cid'}))
    drv_team_yr = drv_team_yr.merge(
        e_cons_nm[['constructorId','_tk']].rename(columns={'constructorId':'_cid'}),
        on='_cid', how='left')

    print(f"  Driver standings: {len(drv_season_stand)} | Cons standings: {len(cons_season_stand)}")
    print(f"  Grid rows: {len(grid_lookup):,} | Quali rows: {len(quali_lookup):,}")
    print("✅ Ergast ready — 30 new features will be added to pipeline A")
else:
    print("⚠️  Ergast not loaded — all Ergast features will be 0")


## Pipeline A — Engineered Features
Used for: **LGB · XGB · CatBoost · RealMLP-A · RealMLP-C**

In [ ]:
%%time
FS_A = {}
COMBO_COLS    = [('Race','Compound'), ('Race','Year'), ('Driver','Compound')]
COMBO_NAMES_A = [f"{c1}_{c2}_" for c1, c2 in COMBO_COLS]

def make_features_A(df, fit=False):
    df = df.copy().sort_values(['Driver','Race','Year','LapNumber']).reset_index(drop=True)
    g       = df.groupby(['Driver','Race','Year'])
    g_stint = df.groupby(['Driver','Race','Year','Stint'])

    # ── A. Tyre ─────────────────────────────────────────────────────────
    df['tyre_life_sq']        = df['TyreLife'] ** 2
    df['tyre_life_log']       = np.log1p(df['TyreLife'])
    df['tyre_life_sqrt']      = np.sqrt(df['TyreLife'])
    df['deg_per_lap']         = df['Cumulative_Degradation'] / (df['TyreLife'] + 1)
    df['compound_life_ratio'] = df['TyreLife'] / (
        df.groupby('Compound')['TyreLife'].transform('max') + 1e-9)

    # ── B. Race progress ────────────────────────────────────────────────
    df['est_total_laps']     = (df['LapNumber'] / (df['RaceProgress'] + 1e-9)).round().clip(30, 80)
    df['laps_remaining']     = (df['est_total_laps'] - df['LapNumber']).clip(lower=0)
    df['tyre_pct_remaining'] = df['TyreLife'] / (df['laps_remaining'] + 1)
    df['is_pit_window']      = ((df['RaceProgress'] >= 0.28) & (df['RaceProgress'] <= 0.62)).astype(int)
    df['is_late_race']       = (df['RaceProgress'] > 0.75).astype(int)
    df['position_pressure']  = df['Position'] * (1 - df['RaceProgress'])
    df['urgency_score']      = df['Cumulative_Degradation'].abs() * (1 - df['RaceProgress'])
    df['race_phase']         = pd.cut(df['RaceProgress'], bins=[0,.25,.5,.75,1.01],
                                      labels=[0,1,2,3]).astype(float)
    df['norm_position']      = 1 - (df['Position'] - 1) / 19.0
    df['life_x_progress']    = df['TyreLife'] * df['RaceProgress']

    # ── C. Lags, rolling & stint ────────────────────────────────────────
    df['delta_lag1']   = g['LapTime_Delta'].shift(1)
    df['delta_lag2']   = g['LapTime_Delta'].shift(2)
    df['prev_pit']     = g['PitStop'].shift(1).fillna(0)
    df['delta_accel']  = df['LapTime_Delta'] - df['delta_lag1']
    df['roll3_lt']     = g['LapTime (s)'].transform(lambda x: x.rolling(3,  min_periods=1).mean())
    df['roll5_lt']     = g['LapTime (s)'].transform(lambda x: x.rolling(5,  min_periods=1).mean())
    df['roll7_lt']     = g['LapTime (s)'].transform(lambda x: x.rolling(7,  min_periods=1).mean())
    df['roll10_lt']    = g['LapTime (s)'].transform(lambda x: x.rolling(10, min_periods=1).mean())
    df['roll15_lt']    = g['LapTime (s)'].transform(lambda x: x.rolling(15, min_periods=1).mean())
    df['roll3_d']      = g['LapTime_Delta'].transform(lambda x: x.rolling(3, min_periods=1).mean())
    df['roll7_d']      = g['LapTime_Delta'].transform(lambda x: x.rolling(7, min_periods=1).mean())
    df['roll3_std']    = g['LapTime (s)'].transform(lambda x: x.rolling(3, min_periods=1).std().fillna(0))
    df['lap_vs_r3']    = df['LapTime (s)'] - df['roll3_lt']
    df['lap_vs_r5']    = df['LapTime (s)'] - df['roll5_lt']
    df['lap_vs_r7']    = df['LapTime (s)'] - df['roll7_lt']
    df['lap_vs_r10']   = df['LapTime (s)'] - df['roll10_lt']
    df['deg_velocity'] = g['Cumulative_Degradation'].transform(lambda x: x.diff(3).fillna(0) / 3)
    df['is_slow_lap']  = (df['LapTime (s)'] > df['roll5_lt'] * 1.15).astype(int)
    df['lap_in_stint']    = g_stint.cumcount()
    df['stint_start_lap'] = g_stint['LapNumber'].transform('min')

    # ── D. Race-level context ────────────────────────────────────────────
    if fit:
        FS_A['pit_laps']    = df[df['PitNextLap']==1].groupby(['Race','Year'])['LapNumber'].mean().rename('race_avg_pit_lap')
        FS_A['total_laps']  = df.groupby(['Race','Year'])['LapNumber'].max().rename('race_total_laps')
        FS_A['comp_life']   = df[df['PitNextLap']==1].groupby('Compound')['TyreLife'].mean().rename('compound_avg_life')
        FS_A['race_stints'] = df.groupby(['Race','Year'])['Stint'].max().rename('race_max_stint')
    df = df.merge(FS_A['pit_laps'].reset_index(),    on=['Race','Year'], how='left')
    df = df.merge(FS_A['total_laps'].reset_index(),  on=['Race','Year'], how='left')
    df = df.merge(FS_A['comp_life'].reset_index(),   on='Compound',      how='left')
    df = df.merge(FS_A['race_stints'].reset_index(), on=['Race','Year'], how='left')
    df['pit_window_flag']      = (np.abs(df['LapNumber'] - df['race_avg_pit_lap'].fillna(35)) <= 3).astype(int)
    df['tyre_vs_comp_avg']     = df['TyreLife'] - df['compound_avg_life'].fillna(25)
    df['overdue_pit']          = (df['TyreLife'] > df['compound_avg_life'].fillna(25)).astype(int)
    df['laps_remaining_race']  = df['race_total_laps'].fillna(60) - df['LapNumber']
    df['tyre_age_pct_race']    = df['TyreLife'] / (df['race_total_laps'].fillna(60) + 1)
    df['stint_progress']       = df['Stint'] / (df['race_max_stint'].fillna(3) + 1)
    df['tyre_life_pct']        = df['TyreLife'] / df['compound_avg_life'].fillna(25).clip(lower=1)
    df['stint_end_est']        = df['stint_start_lap'] + df['compound_avg_life'].fillna(25)
    df['laps_until_stop']      = (df['stint_end_est'] - df['LapNumber']).clip(lower=-20)

    # ── E. Advanced strategy ─────────────────────────────────────────────
    df['cliff_flag']       = ((df['LapTime_Delta'] > df['roll3_d'].fillna(0) + 2*df['roll3_std'].fillna(0))
                              & (df['TyreLife'] > 15)).astype(int)
    df['laps_to_stop']     = df['compound_avg_life'].fillna(25) - df['TyreLife']
    df['past_optimal']     = (df['laps_to_stop'] < -3).astype(int)
    df['must_pit_or_stay'] = (df['laps_remaining_race'] < df['TyreLife'] * 0.4).astype(int)
    df['undercut_threat']  = ((df['Position'] <= 10) & (df['is_pit_window'] == 1)).astype(int)
    df['relative_stint']   = df['Stint'] / df['race_max_stint'].fillna(3).clip(lower=1)

    # ── F. Interactions ──────────────────────────────────────────────────
    df['deg_x_win']   = df['Cumulative_Degradation'] * df['is_pit_window']
    df['over_x_win']  = df['overdue_pit'] * df['is_pit_window']
    df['tyre_x_pres'] = df['TyreLife'] * df['position_pressure']
    df['r3_x_life']   = df['roll3_d'].fillna(0) * df['TyreLife']
    df['comp_stint']  = df['Compound'].map({'SOFT':2,'MEDIUM':1,'HARD':0}).fillna(1) * df['Stint']
    df['lap_div_rp']  = (df['LapNumber'] / (df['RaceProgress'] + 1e-6)).astype('float32')
    df['tl_div_ln']   = (df['TyreLife'] / df['LapNumber'].clip(lower=1)).astype('float32')

    # ── F2. New advanced features ────────────────────────────────────────
    df['lap_accel_smooth']    = g['LapTime_Delta'].transform(
        lambda x: x.rolling(5, min_periods=2).mean().diff().fillna(0))
    df['tyre_cliff_imminent'] = (
        (df['lap_accel_smooth'] > 0.3) & (df['TyreLife'] > 12)
    ).astype(int)

    df['laps_to_window_start'] = (df['race_avg_pit_lap'].fillna(35) * 0.85 - df['LapNumber']).clip(-20, 30)
    df['laps_to_window_end']   = (df['race_avg_pit_lap'].fillna(35) * 1.15 - df['LapNumber']).clip(-20, 30)
    df['in_optimal_window']    = (
        (df['laps_to_window_start'] <= 0) & (df['laps_to_window_end'] >= 0)
    ).astype(int)

    df['stint_lt_baseline'] = g_stint['LapTime (s)'].transform('first')
    df['stint_degradation_ratio'] = (
        (df['LapTime (s)'] - df['stint_lt_baseline']) /
        (df['stint_lt_baseline'] + 1e-9)
    ).clip(-0.1, 0.3)

    if fit:
        FS_A['compound_race_lt'] = (
            df.groupby(['Race', 'Year', 'Compound'])['LapTime (s)']
            .median().rename('compound_race_median_lt')
        )
    df = df.merge(FS_A['compound_race_lt'].reset_index(),
                  on=['Race', 'Year', 'Compound'], how='left')
    df['lap_vs_compound_baseline'] = (
        df['LapTime (s)'] - df['compound_race_median_lt'].fillna(df['LapTime (s)'])
    ).clip(-5, 15)

    df['roll7_std']    = g['LapTime (s)'].transform(
        lambda x: x.rolling(7, min_periods=2).std().fillna(0))
    df['roll3_var_ratio'] = (df['roll3_std'] / (df['roll7_std'] + 1e-9)).clip(0, 5)

    df['pos_lag3']     = g['Position'].shift(3)
    df['pos_trend_3']  = (df['Position'] - df['pos_lag3'].fillna(df['Position'])).clip(-10, 10)
    df['losing_positions'] = (df['pos_trend_3'] > 1).astype(int)

    if fit:
        FS_A['race_compound_max'] = (
            df.groupby(['Race', 'Year', 'Compound'])['TyreLife']
            .max().rename('race_compound_max_life')
        )
    df = df.merge(FS_A['race_compound_max'].reset_index(),
                  on=['Race', 'Year', 'Compound'], how='left')
    df['tyre_freshness_pct'] = (
        1 - df['TyreLife'] / (df['race_compound_max_life'].fillna(40) + 1)
    ).clip(0, 1)

    df['must_pit_signal'] = (
        df['tyre_cliff_imminent'] * df['overdue_pit'] * df['in_optimal_window']
    ).astype(int)
    df['urgency_composite'] = (
        df['urgency_score'] * df['stint_degradation_ratio'].clip(0, 1) *
        (1 + df['tyre_cliff_imminent'])
    )

    if fit:
        FS_A['driver_compound_avg_life'] = (
            df[df['PitNextLap'] == 1]
            .groupby(['Driver', 'Compound'])['TyreLife']
            .mean().rename('dc_avg_stint_life')
        )
    _dc_life = FS_A.get('driver_compound_avg_life')
    if _dc_life is not None:
        df = df.merge(_dc_life.reset_index(), on=['Driver', 'Compound'], how='left')
    else:
        df['dc_avg_stint_life'] = 25.0
    df['driver_vs_avg_life'] = (
        df['TyreLife'] - df['dc_avg_stint_life'].fillna(25)
    ).clip(-20, 20)
    df['driver_overdue_personal'] = (
        df['TyreLife'] > df['dc_avg_stint_life'].fillna(25)
    ).astype(int)

    # ── F3. Field-level competitor features (cross-car context) ─────────────
    g_lap = df.groupby(['Race','Year','LapNumber'])
    df['avg_field_tyre_age']  = g_lap['TyreLife'].transform('mean')
    df['max_field_tyre_age']  = g_lap['TyreLife'].transform('max')
    df['min_field_tyre_age']  = g_lap['TyreLife'].transform('min')
    df['field_tyre_age_pct']  = (df['TyreLife'] / (df['max_field_tyre_age'] + 1)).clip(0, 1)
    df['is_oldest_tyre']      = (df['TyreLife'] == df['max_field_tyre_age']).astype(int)
    df['cars_older_tyres']    = g_lap['TyreLife'].transform(
        lambda x: (x > x.mean()).sum()).astype(int)
    df['n_diff_compounds']    = g_lap['Compound'].transform('nunique').astype(int)
    # Cars that pitted recently (PitStop=1 means they pitted last lap — not target)
    df['n_pitted_this_lap']   = g_lap['PitStop'].transform('sum')
    g_race = df.groupby(['Driver','Race','Year'])
    df['n_pitted_race_last5'] = g_race['PitStop'].transform(
        lambda x: x.shift(1).fillna(0).rolling(5, min_periods=1).sum())
    # Field pit rate context: rolling average pit rate across the race/lap window
    df['field_pit_rate']      = g_lap['PitStop'].transform('mean')
    # Position-adjusted field context: cars behind this car that have newer tyres
    df['tyre_age_vs_field']   = (df['TyreLife'] - df['avg_field_tyre_age']).clip(-20, 20)

    # ── F4. Safety car proxy — now split VSC vs Full SC ──────────────────
    df['field_median_lt']     = g_lap['LapTime (s)'].transform('median')
    _sc_ratio = df['field_median_lt'] / df['roll5_lt'].fillna(df['LapTime (s)']).clip(lower=60)
    df['is_sc_proxy']     = (_sc_ratio > 1.08).astype(int)   # kept for stacking compat
    df['is_vsc_proxy']    = ((_sc_ratio > 1.08) & (_sc_ratio <= 1.30)).astype(int)
    df['is_full_sc_proxy']= (_sc_ratio > 1.30).astype(int)

    def _laps_since_sc(grp):
        out = []; cnt = 99
        for v in grp:
            if v == 1: cnt = 0
            else: cnt += 1
            out.append(min(cnt, 30))
        return out

    df['laps_since_sc']      = g_race['is_sc_proxy'].transform(
        lambda x: _laps_since_sc(x.tolist())).astype(int)
    df['laps_since_vsc']     = g_race['is_vsc_proxy'].transform(
        lambda x: _laps_since_sc(x.tolist())).astype(int)
    df['laps_since_full_sc'] = g_race['is_full_sc_proxy'].transform(
        lambda x: _laps_since_sc(x.tolist())).astype(int)
    df['in_sc_window']       = (df['laps_since_sc'] <= 3).astype(int)
    df['in_vsc_window']      = (df['laps_since_vsc'] <= 2).astype(int)
    df['in_full_sc_window']  = (df['laps_since_full_sc'] <= 5).astype(int)

    # ── F5. Gap-to-car features — sort+shift (memory-efficient, no merge) ──
    PIT_DELTA = 22.0
    df['_clean_lt'] = df['LapTime (s)'] - df['PitStop'].fillna(0) * PIT_DELTA
    df['_cum_rt']   = df.groupby(['Driver','Race','Year'])['_clean_lt'].cumsum()

    # Sort by position within each lap, shift to get car ahead / behind
    _gap = (df[['Race','Year','LapNumber','Position','_cum_rt','PitStop']]
            .sort_values(['Race','Year','LapNumber','Position'])
            .reset_index(drop=False))   # 'index' col = original df row number
    _glap = _gap.groupby(['Race','Year','LapNumber'])
    _gap['_cum_ahead']    = _glap['_cum_rt'].shift(1)   # car at pos-1 (ahead)
    _gap['_cum_behind']   = _glap['_cum_rt'].shift(-1)  # car at pos+1 (behind)
    _gap['_ahead_pitted'] = _glap['PitStop'].shift(1).fillna(0)
    _gap['_behind_pitted']= _glap['PitStop'].shift(-1).fillna(0)
    _gap = _gap.set_index('index')

    df['_cum_ahead']           = _gap['_cum_ahead'].reindex(df.index).values
    df['_cum_behind']          = _gap['_cum_behind'].reindex(df.index).values
    df['car_ahead_pitted_now'] = _gap['_ahead_pitted'].reindex(df.index).fillna(0).values.astype(int)
    df['car_behind_pitted_now']= _gap['_behind_pitted'].reindex(df.index).fillna(0).values.astype(int)
    del _gap, _glap

    df['gap_to_car_ahead']  = (df['_cum_rt'] - df['_cum_ahead']).clip(-5, 60).fillna(30)
    df['gap_to_car_behind'] = (df['_cum_behind'] - df['_cum_rt']).clip(-5, 60).fillna(30)
    df['in_drs_range']      = (df['gap_to_car_ahead'] < 1.2).astype(int)
    df['undercut_viable']   = ((df['gap_to_car_ahead'] > 0.5) & (df['gap_to_car_ahead'] < 4.0)).astype(int)
    df['threat_from_behind']= (df['gap_to_car_behind'] < 2.0).astype(int)
    _g_tmp = df.groupby(['Driver','Race','Year'])
    df['gap_ahead_delta']   = _g_tmp['gap_to_car_ahead'].diff(3).fillna(0).clip(-10, 10)
    df['nearby_pit_pressure']= (
        df['car_ahead_pitted_now'] + df['car_behind_pitted_now'] +
        _g_tmp['car_ahead_pitted_now'].transform(lambda x: x.shift(1).fillna(0)) +
        _g_tmp['car_behind_pitted_now'].transform(lambda x: x.shift(1).fillna(0))
    ).clip(0, 4).astype(int)
    del _g_tmp

    # ── F6. Mandatory compound rule (FIA: must use 2 different dry compounds) ──
    # Vectorized cummax — far more memory efficient than list-based lambda
    df['_comp_first'] = df.groupby(['Driver','Race','Year'])['Compound'].transform('first')
    df['_comp_changed'] = (df['Compound'] != df['_comp_first']).astype(int)
    df['n_compounds_used'] = (
        df.groupby(['Driver','Race','Year'])['_comp_changed'].transform('cummax') + 1
    ).astype(int)
    df.drop(columns=['_comp_first','_comp_changed'], inplace=True, errors='ignore')
    df['mandatory_pit_pending'] = (
        (df['n_compounds_used'] < 2) & (df['RaceProgress'] > 0.45)
    ).astype(int)
    df['mandatory_urgency'] = (df['mandatory_pit_pending'] * df['RaceProgress']).astype('float32')

    # ── F7. Fuel load correction — cleaner tyre degradation signal ───────
    FUEL_GAIN_PER_LAP = 0.035
    df['fuel_adj_lt']        = df['LapTime (s)'] + df['LapNumber'] * FUEL_GAIN_PER_LAP
    df['fuel_corrected_deg'] = (df['fuel_adj_lt']
        - df.groupby(['Driver','Race','Year'])['fuel_adj_lt']
                                .transform('first')).clip(-5, 20)

    df.drop(columns=['_clean_lt','_cum_rt','_cum_ahead','_cum_behind','fuel_adj_lt'],
            inplace=True, errors='ignore')

    # ── G. Ordinal / year ────────────────────────────────────────────────
    df['compound_ord'] = df['Compound'].map({'SOFT':2,'MEDIUM':1,'HARD':0,'INTERMEDIATE':3,'WET':4}).fillna(1)
    df['year_le']      = df['Year'] - df['Year'].min()

    # ── H. Numeric → category (for RealMLP embeddings) ──────────────────
    base_nums = ['PitStop','LapNumber','Stint','TyreLife','Position',
                 'LapTime (s)','LapTime_Delta','Cumulative_Degradation',
                 'RaceProgress','Position_Change']
    for col in base_nums:
        cname = f"{col}_cat_"
        if fit:
            codes, uniques = np.floor(df[col]).factorize()
            FS_A[f'nc_{col}'] = {v: i for i, v in enumerate(uniques)}
        df[cname] = np.floor(df[col]).map(FS_A[f'nc_{col}']).fillna(-1).astype('int32').astype(str)
    if fit:
        kb = KBinsDiscretizer(n_bins=200, encode='ordinal', strategy='quantile', subsample=None)
        df['rp_bin_'] = kb.fit_transform(df[['RaceProgress']]).ravel().astype('int32').astype(str)
        FS_A['kb'] = kb
    else:
        df['rp_bin_'] = FS_A['kb'].transform(df[['RaceProgress']]).ravel().astype('int32').astype(str)
    for c1, c2 in COMBO_COLS:
        cn = f"{c1}_{c2}_"
        combo = df[c1].astype(str) + '_' + df[c2].astype(str)
        if fit:
            codes, uniques = combo.factorize()
            FS_A[cn] = {v: i for i, v in enumerate(uniques)}
        df[cn] = combo.map(FS_A[cn]).fillna(-1).astype('int32').astype(str)

    # ── I. Target encoding (OOF + interaction TEs) ───────────────────────
    from sklearn.model_selection import StratifiedKFold as _SKF
    if fit and 'PitNextLap' in df.columns:
        teskf = _SKF(5, shuffle=True, random_state=SEED)
        gm = df['PitNextLap'].mean()
        for col in ['Driver', 'Race']:
            df[f'{col}_te'] = 0.0
            for ti, vi in teskf.split(df, df['PitNextLap']):
                m = df.iloc[ti].groupby(col)['PitNextLap'].mean()
                df.iloc[vi, df.columns.get_loc(f'{col}_te')] = (
                    df.iloc[vi][col].map(m).fillna(gm).values)
            FS_A[f'{col}_te_map'] = df.groupby(col)['PitNextLap'].mean().to_dict()
        for (c1, c2), te_name in [
            (('Driver', 'Compound'), 'Driver_Compound_te'),
            (('Race',   'Compound'), 'Race_Compound_te'),
            (('Driver', 'Race'),     'Driver_Race_te'),
        ]:
            combined = (df[c1].astype(str) + '_' + df[c2].astype(str)).values
            df[te_name] = gm
            for ti, vi in teskf.split(df, df['PitNextLap']):
                m = (pd.Series(df['PitNextLap'].values[ti], index=combined[ti])
                       .groupby(level=0).mean())
                df.iloc[vi, df.columns.get_loc(te_name)] = (
                    pd.Series(combined[vi]).map(m).fillna(gm).values)
            FS_A[f'{te_name}_map'] = (
                pd.Series(df['PitNextLap'].values, index=combined)
                  .groupby(level=0).mean().to_dict())
    else:
        for col in ['Driver', 'Race']:
            gm = float(np.mean(list(FS_A.get(f'{col}_te_map', {}).values()) or [0.2]))
            df[f'{col}_te'] = df[col].map(FS_A.get(f'{col}_te_map', {})).fillna(gm)
        for (c1, c2), te_name in [
            (('Driver', 'Compound'), 'Driver_Compound_te'),
            (('Race',   'Compound'), 'Race_Compound_te'),
            (('Driver', 'Race'),     'Driver_Race_te'),
        ]:
            combined = (df[c1].astype(str) + '_' + df[c2].astype(str)).values
            gm = float(np.mean(list(FS_A.get(f'{te_name}_map', {}).values()) or [0.2]))
            df[te_name] = pd.Series(combined).map(FS_A.get(f'{te_name}_map', {})).fillna(gm).values

    # ── Ergast feature join ────────────────────────────────────────────────
    df['_dk'] = df['Driver'].map(norm_driver)
    df['_rk'] = df['Race'].map(norm_circuit)
    df['_yr'] = df['Year'].astype(int)

    if HAS_ERGAST:
        # Grid position (race-level join)
        _grid_m = df[['_dk','_rk','_yr']].merge(grid_lookup, on=['_dk','_rk','_yr'], how='left')
        df['grid_position']  = _grid_m['grid'].fillna(10).values
        df['grid_vs_pos']    = (df['Position'] - df['grid_position']).clip(-15, 15)
        df['started_front5'] = (df['grid_position'] <= 5).astype(int)
        df['started_back10'] = (df['grid_position'] >= 11).astype(int)

        # Qualifying position
        _qual_m = df[['_dk','_rk','_yr']].merge(quali_lookup, on=['_dk','_rk','_yr'], how='left')
        df['quali_position'] = _qual_m['quali_pos'].fillna(10).values
        df['grid_penalty']   = (df['grid_position'] - df['quali_position']).clip(-5, 15)

        # Circuit pit stop priors (Ergast historical data — much more data than debashish)
        _ckt_p = df[['_rk']].merge(ckt_pit_stats, on='_rk', how='left')
        df['ckt_pit_avg_erg']   = _ckt_p['ckt_pit_avg'].fillna(28.0).values
        df['ckt_pit_std_erg']   = _ckt_p['ckt_pit_std'].fillna(8.0).values
        df['laps_vs_ckt_erg']   = (df['TyreLife'] - df['ckt_pit_avg_erg']).clip(-25, 25)
        df['in_ckt_window_erg'] = (df['laps_vs_ckt_erg'].abs() <= df['ckt_pit_std_erg']).astype(int)

        # Driver pit stop priors
        _drv_p = df[['_dk']].merge(drv_pit_stats, on='_dk', how='left')
        df['drv_pit_avg_erg']     = _drv_p['drv_pit_avg'].fillna(30.0).values
        df['drv_pit_std_erg']     = _drv_p['drv_pit_std'].fillna(7.0).values
        df['laps_vs_drv_erg']     = (df['TyreLife'] - df['drv_pit_avg_erg']).clip(-25, 25)
        df['drv_hist_overdue_erg']= (df['TyreLife'] > df['drv_pit_avg_erg']).astype(int)

        # Circuit degradation rate from lapTimes (ms per lap)
        df['ckt_deg_rate'] = df['_rk'].map(
            lambda k: ckt_deg_rates.get(k, {}).get('ckt_deg_rate_ms_per_lap', 0.0))
        df['ckt_deg_r2']   = df['_rk'].map(
            lambda k: ckt_deg_rates.get(k, {}).get('ckt_deg_r2', 0.0))
        df['expected_lt_loss'] = (df['ckt_deg_rate'] * df['TyreLife'] / 1000.0).clip(0, 5)

        # Driver championship (prior year — no leakage)
        _yr_prev = (df['_yr'] - 1).astype(int)
        _dstand = pd.DataFrame({'_dk': df['_dk'].values, '_yr': _yr_prev.values})
        _dstand = _dstand.merge(drv_season_stand, on=['_dk','_yr'], how='left')
        df['drv_champ_pos']   = _dstand['drv_season_pos'].fillna(10).values
        df['drv_champ_pts']   = _dstand['drv_season_pts'].fillna(0).values
        df['drv_champ_front'] = (df['drv_champ_pos'] <= 3).astype(int)
        df['drv_champ_back']  = (df['drv_champ_pos'] >= 8).astype(int)

        # Constructor championship (prior year, via driver→team lookup)
        _team = pd.DataFrame({'_dk': df['_dk'].values, '_yr': df['_yr'].values})
        _team = _team.merge(drv_team_yr[['_dk','_yr','_tk']], on=['_dk','_yr'], how='left')
        _team_prev = pd.DataFrame({'_tk': _team['_tk'].values, '_yr': _yr_prev.values})
        _cstand = _team_prev.merge(cons_season_stand, on=['_tk','_yr'], how='left')
        df['cons_champ_pos'] = _cstand['cons_season_pos'].fillna(5).values
        df['cons_champ_pts'] = _cstand['cons_season_pts'].fillna(0).values
        df['cons_top3']      = (df['cons_champ_pos'] <= 3).astype(int)

        # Combined urgency + window features
        df['ckt_drv_urgency'] = df['in_ckt_window_erg'] * df['drv_hist_overdue_erg']
        df['past_ckt_window'] = (df['laps_vs_ckt_erg'] - df['ckt_pit_std_erg']).clip(0, 20)

        # ── Change 3: 3 new high-signal interaction features ──────────────
        # High-degradation circuits punish long stints much harder
        df['deg_rate_x_life']        = (df['ckt_deg_rate'].clip(0, 5) * df['TyreLife']).clip(0, 100)
        # Pit urgency: past window AND driver overdue AND circuit says stop
        df['pit_urgency_erg']        = (df['past_ckt_window'] * (1 + df['overdue_pit'])
                                        * (1 + df['ckt_drv_urgency'])).clip(0, 50)
        # Is tyre degrading FASTER than this circuit typically degrades?
        df['lt_worse_than_expected'] = (df['lap_vs_r3'] - df['expected_lt_loss']).clip(-3, 10)
        df['tyre_degrading_faster']  = (df['lt_worse_than_expected'] > 0.5).astype(int)

    else:
        for _c in ['grid_position','grid_vs_pos','started_front5','started_back10',
                   'quali_position','grid_penalty',
                   'ckt_pit_avg_erg','ckt_pit_std_erg','laps_vs_ckt_erg','in_ckt_window_erg',
                   'drv_pit_avg_erg','drv_pit_std_erg','laps_vs_drv_erg','drv_hist_overdue_erg',
                   'ckt_deg_rate','ckt_deg_r2','expected_lt_loss',
                   'drv_champ_pos','drv_champ_pts','drv_champ_front','drv_champ_back',
                   'cons_champ_pos','cons_champ_pts','cons_top3',
                   'ckt_drv_urgency','past_ckt_window',
                   'deg_rate_x_life','pit_urgency_erg','lt_worse_than_expected','tyre_degrading_faster']:
            df[_c] = 0.0

    df.drop(columns=['_dk','_rk','_yr'], inplace=True, errors='ignore')

    return df.fillna(0)

print("Engineering train A...")
train_A = make_features_A(train, fit=True)
print("Engineering test A...")
test_A  = make_features_A(test,  fit=False)
if HAS_ORIG:
    print("Engineering original A...")
    orig_A = make_features_A(orig, fit=False)
else:
    orig_A = None

CAT_COLS_A = [c for c in train_A.columns if c.endswith('_')]
DROP_A     = ['id','Driver','Compound','Race','Year','PitNextLap'] + CAT_COLS_A
TREE_FEATS = [c for c in train_A.columns if c not in DROP_A]
MLP_FEATS_A= TREE_FEATS + CAT_COLS_A

X      = train_A[TREE_FEATS].astype(np.float32)
y      = train_A['PitNextLap'].astype(int)
X_test = test_A[TREE_FEATS].astype(np.float32)

if orig_A is not None:
    X_orig = orig_A.reindex(columns=TREE_FEATS, fill_value=0).astype(np.float32)
    y_orig = orig_A['PitNextLap'].astype(int)
else:
    X_orig = y_orig = None

skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)

def prep_mlp(df, feats, cat_cols):
    d = df.reindex(columns=feats, fill_value=0).copy()
    for c in cat_cols:
        if c in d.columns: d[c] = d[c].astype('category')
    return d

X_mlp_A      = prep_mlp(train_A, MLP_FEATS_A, CAT_COLS_A)
X_test_mlp_A = prep_mlp(test_A,  MLP_FEATS_A, CAT_COLS_A)
X_orig_mlp_A = prep_mlp(orig_A,  MLP_FEATS_A, CAT_COLS_A) if orig_A is not None else None

print(f"Tree feats: {len(TREE_FEATS)} | MLP-A feats: {len(MLP_FEATS_A)}")


## Pipeline B — Raw Features
Used for: **RealMLP-B**

In [ ]:
%%time
FS_B = {}
COMBO_NAMES_B = [f"{c1}_{c2}_b_" for c1, c2 in COMBO_COLS]

def make_features_B(df, fit=False):
    df = df.copy().sort_values(['Driver','Race','Year','LapNumber']).reset_index(drop=True)
    base_num = ['PitStop','LapNumber','Stint','TyreLife','Position',
                'LapTime (s)','LapTime_Delta','Cumulative_Degradation',
                'RaceProgress','Position_Change']
    for col in base_num:
        cn = f"{col}_bcat_"
        if fit:
            codes, uniques = np.floor(df[col]).factorize()
            FS_B[f'nc_{col}'] = {v: i for i, v in enumerate(uniques)}
        df[cn] = np.floor(df[col]).map(FS_B[f'nc_{col}']).fillna(-1).astype('int32').astype(str)
    df['_b_lap_div_rp'] = (df['LapNumber'] / (df['RaceProgress'] + 1e-6)).astype('float32')
    df['_b_tl_div_ln']  = (df['TyreLife']  / df['LapNumber'].clip(lower=1)).astype('float32')
    if fit:
        kb = KBinsDiscretizer(n_bins=200, encode='ordinal', strategy='quantile', subsample=None)
        df['rp200_b_'] = kb.fit_transform(df[['RaceProgress']]).ravel().astype('int32').astype(str)
        FS_B['kb'] = kb
    else:
        df['rp200_b_'] = FS_B['kb'].transform(df[['RaceProgress']]).ravel().astype('int32').astype(str)
    for (c1, c2), cn in zip(COMBO_COLS, COMBO_NAMES_B):
        combo = df[c1].astype(str) + '_' + df[c2].astype(str)
        if fit:
            codes, uniques = combo.factorize()
            FS_B[cn] = {v: i for i, v in enumerate(uniques)}
        df[cn] = combo.map(FS_B[cn]).fillna(-1).astype('int32').astype(str)
    return df

print("Engineering train B...")
train_B = make_features_B(train, fit=True)
print("Engineering test B...")
test_B  = make_features_B(test,  fit=False)
orig_B  = make_features_B(orig, fit=False) if HAS_ORIG else None

CAT_COLS_B  = [c for c in train_B.columns if c.endswith('_b_') or c == 'rp200_b_' or c.endswith('_bcat_')]
NUM_COLS_B  = [c for c in train_B.columns if c.startswith('_b_')]
RAW_NUM_B   = ['PitStop','LapNumber','Stint','TyreLife','Position',
               'LapTime (s)','LapTime_Delta','Cumulative_Degradation',
               'RaceProgress','Position_Change']
MLP_FEATS_B = RAW_NUM_B + NUM_COLS_B + CAT_COLS_B

def prep_mlp_b(df, feats, cat_cols):
    d = df.reindex(columns=feats, fill_value=0).copy()
    for c in cat_cols:
        if c in d.columns: d[c] = d[c].astype('category')
    return d

X_mlp_B      = prep_mlp_b(train_B, MLP_FEATS_B, CAT_COLS_B)
X_test_mlp_B = prep_mlp_b(test_B,  MLP_FEATS_B, CAT_COLS_B)
X_orig_mlp_B = prep_mlp_b(orig_B,  MLP_FEATS_B, CAT_COLS_B) if orig_B is not None else None
print(f"MLP-B feats: {len(MLP_FEATS_B)}")


## LGB Optuna Tuning (warm-start, 6 min, 144-feature search space)

In [ ]:
%%time
LGB_DEVICE = 'gpu' if DEVICE == 'cuda' else 'cpu'

_splits   = list(skf.split(X, y))
ti0_lgb, vi0_lgb = _splits[0]
rng_lgb   = np.random.RandomState(SEED + 1)
sub_lgb   = rng_lgb.choice(ti0_lgb, size=int(len(ti0_lgb) * 0.50), replace=False)
X_tr0_lgb, y_tr0_lgb = X.iloc[sub_lgb], y.iloc[sub_lgb]
X_va0_lgb, y_va0_lgb = X.iloc[vi0_lgb], y.iloc[vi0_lgb]

def lgb_objective(trial):
    params = dict(
        objective='binary', metric='auc', n_estimators=2000,
        learning_rate=trial.suggest_float('lr', 0.01, 0.05, log=True),
        num_leaves=trial.suggest_int('num_leaves', 127, 511),
        min_child_samples=trial.suggest_int('min_child_samples', 15, 60),
        feature_fraction=trial.suggest_float('feature_fraction', 0.30, 0.70),
        bagging_fraction=trial.suggest_float('bagging_fraction', 0.70, 1.0),
        bagging_freq=1,
        lambda_l1=trial.suggest_float('lambda_l1', 0.5, 5.0, log=True),
        lambda_l2=trial.suggest_float('lambda_l2', 1.0, 8.0, log=True),
        max_depth=trial.suggest_int('max_depth', 8, 12),
        path_smooth=trial.suggest_float('path_smooth', 0.0, 0.3),
        min_data_in_bin=trial.suggest_int('min_data_in_bin', 5, 20),
        device=LGB_DEVICE, random_state=SEED, n_jobs=-1, verbose=-1,
    )
    m = lgb.LGBMClassifier(**params)
    m.fit(X_tr0_lgb, y_tr0_lgb, eval_set=[(X_va0_lgb, y_va0_lgb)],
          callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
    return roc_auc_score(y_va0_lgb, m.predict_proba(X_va0_lgb)[:, 1])

lgb_study = optuna.create_study(direction='maximize',
                                 sampler=optuna.samplers.TPESampler(seed=SEED))
# Warm-start: enqueue proven params as trial 0 so we never go backwards
lgb_study.enqueue_trial({
    'lr': 0.025, 'num_leaves': 255, 'min_child_samples': 25,
    'feature_fraction': 0.50, 'bagging_fraction': 0.85,
    'lambda_l1': 1.2, 'lambda_l2': 2.5, 'max_depth': 10,
    'path_smooth': 0.1, 'min_data_in_bin': 10,
})
lgb_study.optimize(lgb_objective, timeout=6*60, show_progress_bar=True)

_lbp = lgb_study.best_params
best_lgb_params = dict(
    objective='binary', metric='auc', n_estimators=6000,
    learning_rate=_lbp['lr'],
    num_leaves=_lbp['num_leaves'],
    min_child_samples=_lbp['min_child_samples'],
    min_data_in_bin=_lbp['min_data_in_bin'],
    feature_fraction=_lbp['feature_fraction'],
    bagging_fraction=_lbp['bagging_fraction'],
    bagging_freq=1,
    lambda_l1=_lbp['lambda_l1'],
    lambda_l2=_lbp['lambda_l2'],
    max_depth=_lbp['max_depth'],
    path_smooth=_lbp['path_smooth'],
    device=LGB_DEVICE, random_state=SEED, n_jobs=-1, verbose=-1,
)
print(f"LGB best val AUC: {lgb_study.best_value:.5f}  ({len(lgb_study.trials)} trials)")
print(f"Best params: {lgb_study.best_params}")


## Model 1 — LightGBM Training ~12 min

In [ ]:
%%time
oof_lgb  = np.zeros(len(X))
pred_lgb = np.zeros(len(X_test))
lgb_models = []

for fold, (ti, vi) in enumerate(skf.split(X, y)):
    Xtr, Xva, ytr, yva = X.iloc[ti], X.iloc[vi], y.iloc[ti], y.iloc[vi]
    if X_orig is not None:
        Xtr = pd.concat([Xtr, X_orig], ignore_index=True)
        ytr = pd.concat([ytr, y_orig], ignore_index=True)
    m = lgb.LGBMClassifier(**best_lgb_params)
    m.fit(Xtr, ytr, eval_set=[(Xva, yva)],
          callbacks=[lgb.early_stopping(150, verbose=False), lgb.log_evaluation(1000)])
    oof_lgb[vi]  = m.predict_proba(Xva)[:, 1]
    pred_lgb    += m.predict_proba(X_test)[:, 1] / FOLDS
    lgb_models.append(m)
    print(f"  Fold {fold+1}: {roc_auc_score(yva, oof_lgb[vi]):.5f}")

lgb_auc = roc_auc_score(y, oof_lgb)
print(f"\n{Fore.GREEN}{Style.BRIGHT}LGB OOF: {lgb_auc:.5f}{Style.RESET_ALL}")


## XGBoost Optuna Tuning ~4 min
Single validation fold (fold 0), 50% subsample. 4-minute timeout.

In [ ]:
%%time
_splits = list(skf.split(X, y))
ti0, vi0 = _splits[0]
rng = np.random.RandomState(SEED)
sub_idx = rng.choice(ti0, size=int(len(ti0) * 0.50), replace=False)
X_tr0, y_tr0 = X.iloc[sub_idx], y.iloc[sub_idx]
X_va0, y_va0 = X.iloc[vi0],     y.iloc[vi0]

_xgb_device = 'cuda' if DEVICE == 'cuda' else 'cpu'

def xgb_objective(trial):
    gp = trial.suggest_categorical('grow_policy', ['depthwise', 'lossguide'])
    params = dict(
        n_estimators=1500,
        learning_rate=trial.suggest_float('lr', 0.01, 0.15, log=True),
        max_depth=trial.suggest_int('max_depth', 5, 10),
        min_child_weight=trial.suggest_int('min_child_weight', 1, 10),
        subsample=trial.suggest_float('subsample', 0.5, 1.0),
        colsample_bytree=trial.suggest_float('colsample_bytree', 0.4, 0.9),
        colsample_bylevel=trial.suggest_float('colsample_bylevel', 0.4, 1.0),
        gamma=trial.suggest_float('gamma', 0.0, 2.0),
        reg_alpha=trial.suggest_float('reg_alpha', 0.05, 5.0, log=True),
        reg_lambda=trial.suggest_float('reg_lambda', 0.05, 5.0, log=True),
        grow_policy=gp,
        tree_method='hist', device=_xgb_device,
        eval_metric='auc', early_stopping_rounds=40,
        random_state=SEED, n_jobs=-1, verbosity=0,
    )
    if gp == 'lossguide':
        params['max_leaves'] = trial.suggest_int('max_leaves', 64, 512)
    m = xgb.XGBClassifier(**params)
    m.fit(X_tr0, y_tr0, eval_set=[(X_va0, y_va0)], verbose=False)
    return roc_auc_score(y_va0, m.predict_proba(X_va0)[:, 1])

study = optuna.create_study(direction='maximize',
                             sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(xgb_objective, timeout=4*60, show_progress_bar=True)

_bp = study.best_params
best_xgb_params = dict(
    n_estimators=5000,
    learning_rate=_bp['lr'],
    max_depth=_bp['max_depth'],
    min_child_weight=_bp['min_child_weight'],
    subsample=_bp['subsample'],
    colsample_bytree=_bp['colsample_bytree'],
    colsample_bylevel=_bp['colsample_bylevel'],
    gamma=_bp['gamma'],
    reg_alpha=_bp['reg_alpha'],
    reg_lambda=_bp['reg_lambda'],
    grow_policy=_bp['grow_policy'],
    tree_method='hist', device=_xgb_device,
    eval_metric='auc', early_stopping_rounds=150,
    random_state=SEED, n_jobs=-1,
)
if _bp['grow_policy'] == 'lossguide':
    best_xgb_params['max_leaves'] = _bp['max_leaves']

print(f"Optuna best val AUC: {study.best_value:.5f}  ({len(study.trials)} trials)")
print(f"Best params: {study.best_params}")


## Model 2 — XGBoost (GPU, Optuna-tuned) ~3 min

In [ ]:
%%time
oof_xgb  = np.zeros(len(X))
pred_xgb = np.zeros(len(X_test))

for fold, (ti, vi) in enumerate(skf.split(X, y)):
    Xtr, Xva, ytr, yva = X.iloc[ti], X.iloc[vi], y.iloc[ti], y.iloc[vi]
    if X_orig is not None:
        Xtr = pd.concat([Xtr, X_orig], ignore_index=True)
        ytr = pd.concat([ytr, y_orig], ignore_index=True)
    m = xgb.XGBClassifier(**best_xgb_params)
    m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=500)
    oof_xgb[vi]  = m.predict_proba(Xva)[:, 1]
    pred_xgb    += m.predict_proba(X_test)[:, 1] / FOLDS
    print(f"  Fold {fold+1}: {roc_auc_score(yva, oof_xgb[vi]):.5f}")

xgb_auc = roc_auc_score(y, oof_xgb)
print(f"\n{Fore.GREEN}{Style.BRIGHT}XGB OOF: {xgb_auc:.5f}{Style.RESET_ALL}")


## Model 3 — GRU Sequence Model (T4x2) ~12 min
Processes lap sequences (Driver×Race×Year). Sees the entire stint trajectory, not just current snapshot.

In [ ]:
%%time
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

class LapSequenceDataset(Dataset):
    def __init__(self, df, features, target_col=None, max_len=70):
        self.sequences, self.targets, self.indices = [], [], []
        for _, grp in df.groupby(['Driver','Race','Year']):
            grp = grp.sort_values('LapNumber')
            seq = grp[features].values.astype(np.float32)
            idx = grp.index.tolist()
            T = len(seq)
            if T > max_len:
                seq = seq[-max_len:]; idx = idx[-max_len:]
            elif T < max_len:
                pad = np.zeros((max_len - T, seq.shape[1]), dtype=np.float32)
                seq = np.vstack([pad, seq]); idx = [None] * (max_len - T) + idx
            self.sequences.append(seq); self.indices.append(idx)
            if target_col is not None:
                tgt = grp[target_col].values.astype(np.float32)
                if T > max_len:   tgt = tgt[-max_len:]
                elif T < max_len: tgt = np.concatenate([np.full(max_len - T, -1.0), tgt])
                self.targets.append(tgt)

    def __len__(self): return len(self.sequences)
    def __getitem__(self, i):
        s = torch.tensor(self.sequences[i])
        return (s, torch.tensor(self.targets[i])) if self.targets else s

class GRUPitPredictor(nn.Module):
    def __init__(self, input_dim, hidden=256, layers=2, dropout=0.25):
        super().__init__()
        self.norm = nn.LayerNorm(input_dim)
        self.gru  = nn.GRU(input_dim, hidden, layers, batch_first=True,
                           dropout=dropout if layers > 1 else 0.0)
        self.head = nn.Sequential(
            nn.Linear(hidden, 128), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(128, 64),    nn.GELU(), nn.Dropout(0.1),
            nn.Linear(64, 1))
    def forward(self, x):
        h, _ = self.gru(self.norm(x))
        return self.head(h).squeeze(-1)   # (B, T)

GRU_FEATS  = TREE_FEATS
MAX_LEN    = 70
BATCH_GRU  = 128
GRU_EPOCHS = 15
GRU_LR     = 3e-4

gru_scaler   = StandardScaler()
X_gru_scaled = gru_scaler.fit_transform(train_A[GRU_FEATS].values.astype(np.float32))
X_gru_te_sc  = gru_scaler.transform(test_A[GRU_FEATS].values.astype(np.float32))

train_A_gru = train_A.copy(); train_A_gru[GRU_FEATS] = X_gru_scaled
test_A_gru  = test_A.copy();  test_A_gru[GRU_FEATS]  = X_gru_te_sc
# carry target column through for Dataset
train_A_gru['PitNextLap'] = train_A['PitNextLap'].values

oof_gru  = np.zeros(len(train_A))
pred_gru = np.zeros(len(test_A))
_te_ds   = LapSequenceDataset(test_A_gru, GRU_FEATS, None, MAX_LEN)

for fold, (ti, vi) in enumerate(skf.split(X, y)):
    print(f"### GRU Fold {fold+1}/{FOLDS}")
    tr_df = train_A_gru.iloc[ti].copy()
    va_df = train_A_gru.iloc[vi].copy()

    tr_ds = LapSequenceDataset(tr_df, GRU_FEATS, 'PitNextLap', MAX_LEN)
    va_ds = LapSequenceDataset(va_df, GRU_FEATS, 'PitNextLap', MAX_LEN)

    tr_dl = DataLoader(tr_ds, batch_size=BATCH_GRU, shuffle=True,  num_workers=2, pin_memory=True)
    va_dl = DataLoader(va_ds, batch_size=BATCH_GRU, shuffle=False, num_workers=2, pin_memory=True)
    te_dl = DataLoader(_te_ds, batch_size=BATCH_GRU, shuffle=False, num_workers=2, pin_memory=True)

    model = GRUPitPredictor(len(GRU_FEATS)).to(DEVICE)
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)

    opt   = torch.optim.AdamW(model.parameters(), lr=GRU_LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=GRU_LR, epochs=GRU_EPOCHS,
        steps_per_epoch=len(tr_dl), pct_start=0.1)
    crit  = nn.BCEWithLogitsLoss(reduction='none')

    best_auc, best_state, no_improve = 0.0, None, 0
    PATIENCE = 4

    for epoch in range(GRU_EPOCHS):
        model.train()
        for seqs, tgts in tr_dl:
            seqs, tgts = seqs.to(DEVICE), tgts.to(DEVICE)
            opt.zero_grad()
            logits = model(seqs)
            mask   = (tgts >= 0)
            loss   = (crit(logits, tgts) * mask).sum() / (mask.sum() + 1e-9)
            loss.backward(); opt.step(); sched.step()

        # ── Validation: map predictions back to original row indices ───────
        model.eval()
        fold_preds = {}
        with torch.no_grad():
            for b_i, (seqs, _) in enumerate(va_dl):
                logits_b = torch.sigmoid(model(seqs.to(DEVICE))).cpu().numpy()  # (B, T)
                for b in range(logits_b.shape[0]):
                    seq_i = b_i * BATCH_GRU + b
                    for t, orig_idx in enumerate(va_ds.indices[seq_i]):
                        if orig_idx is not None:
                            fold_preds[orig_idx] = float(logits_b[b, t])
        epoch_preds  = np.array([fold_preds.get(i, 0.0) for i in vi])
        epoch_auc    = roc_auc_score(y.iloc[vi], epoch_preds)
        if epoch_auc > best_auc:
            best_auc   = epoch_auc
            best_state = {k: v.cpu().clone() for k, v in
                          (model.module if hasattr(model,'module') else model).state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= PATIENCE: break
        print(f"  Epoch {epoch+1}: val AUC={epoch_auc:.5f}")

    # ── Write OOF for this fold ────────────────────────────────────────────
    _m = GRUPitPredictor(len(GRU_FEATS)).to(DEVICE)
    _m.load_state_dict(best_state); _m.eval()
    with torch.no_grad():
        for b_i, (seqs, _) in enumerate(va_dl):
            logits_b = torch.sigmoid(_m(seqs.to(DEVICE))).cpu().numpy()
            for b in range(logits_b.shape[0]):
                seq_i = b_i * BATCH_GRU + b
                for t, orig_idx in enumerate(va_ds.indices[seq_i]):
                    if orig_idx is not None:
                        oof_gru[orig_idx] = float(logits_b[b, t])

    # ── Test predictions (accumulated across folds) ────────────────────────
    with torch.no_grad():
        for b_i, seqs in enumerate(te_dl):
            logits_b = torch.sigmoid(_m(seqs.to(DEVICE))).cpu().numpy()
            for b in range(logits_b.shape[0]):
                seq_i = b_i * BATCH_GRU + b
                for t, orig_idx in enumerate(_te_ds.indices[seq_i]):
                    if orig_idx is not None:
                        pred_gru[orig_idx] += float(logits_b[b, t]) / FOLDS

    print(f"  Fold {fold+1} best AUC: {best_auc:.5f}")
    torch.cuda.empty_cache(); gc.collect()

gru_auc = roc_auc_score(y, oof_gru)
print(f"\n{Fore.GREEN}{Style.BRIGHT}GRU OOF: {gru_auc:.5f}{Style.RESET_ALL}")


## Extra Training Data
Original f1_strategy_dataset_v4 appended to MLP training when available.

In [ ]:
# Use the original dataset as extra training data for RealMLP
# Pseudo-labeling was removed (positive rate structurally too low at ~0.8%)
X_extra_mlp_A = X_orig_mlp_A if orig_A is not None else pd.DataFrame()
X_extra_mlp_B = X_orig_mlp_B if orig_B is not None else pd.DataFrame()
y_extra = y_orig if HAS_ORIG else pd.Series(dtype=int)
print(f"Extra training rows (original dataset): {len(y_extra):,}")


## Model 4 — RealMLP-A: Engineered + pseudo-labels ~8 min

In [ ]:
%%time
MLP_PARAMS = dict(
    random_state=SEED, verbosity=1, val_metric_name='1-auc_ovr',
    n_ens=16, n_epochs=4, batch_size=256, use_early_stopping=False,
    lr=0.075, wd=0.018, sq_mom=0.98, lr_sched='cos_anneal',
    first_layer_lr_factor=0.25, embedding_size=6, max_one_hot_cat_size=18,
    hidden_sizes=[512, 256, 128], act='silu', p_drop=0.05, p_drop_sched='expm4t',
    plr_hidden_1=16, plr_hidden_2=8, plr_act_name='gelu',
    plr_lr_factor=0.1151, plr_sigma=2.33, ls_eps=0.01, ls_eps_sched='sqrt_cos',
    add_front_scale=False, bias_init_mode='neg-uniform-dynamic-2',
    tfms=['one_hot','median_center','robust_scale','smooth_clip','embedding','l2_normalize'],
)

oof_mlp_A  = np.zeros(len(X))
pred_mlp_A = np.zeros(len(X_test))

for fold, (ti, vi) in enumerate(skf.split(X, y), 1):
    print(f"### Fold {fold}/{FOLDS}")
    Xtr = X_mlp_A.iloc[ti].copy(); ytr = y.iloc[ti]
    Xva = X_mlp_A.iloc[vi].copy(); yva = y.iloc[vi]
    Xte = X_test_mlp_A.copy()
    Xtr = pd.concat([Xtr, X_extra_mlp_A], ignore_index=True)
    ytr = pd.concat([ytr, y_extra],        ignore_index=True)
    te      = TargetEncoder(cv=FOLDS, smooth='auto', shuffle=True, random_state=SEED)
    te_enc  = te.fit_transform(Xtr[COMBO_NAMES_A], ytr)
    te_names= [f'_te_{c}' for c in COMBO_NAMES_A]
    Xtr[te_names] = te_enc
    Xva[te_names] = te.transform(Xva[COMBO_NAMES_A])
    Xte[te_names] = te.transform(Xte[COMBO_NAMES_A])
    m = RealMLP_TD_Classifier(**MLP_PARAMS)
    m.fit(Xtr, ytr, Xva, yva)
    oof_mlp_A[vi]  = m.predict_proba(Xva)[:, 1]
    pred_mlp_A    += m.predict_proba(Xte)[:, 1] / FOLDS
    print(f"{Fore.GREEN}{Style.BRIGHT}Fold {fold} AUC: {roc_auc_score(yva, oof_mlp_A[vi]):.5f}{Style.RESET_ALL}")
    torch.cuda.empty_cache(); gc.collect()

mlp_A_auc = roc_auc_score(y, oof_mlp_A)
print(f"\n{Fore.GREEN}{Style.BRIGHT}RealMLP-A OOF: {mlp_A_auc:.5f}{Style.RESET_ALL}")


## Model 5 — RealMLP-B: Raw + pseudo-labels ~8 min

In [ ]:
%%time
oof_mlp_B  = np.zeros(len(X))
pred_mlp_B = np.zeros(len(X_test))

for fold, (ti, vi) in enumerate(skf.split(X, y), 1):
    print(f"### Fold {fold}/{FOLDS}")
    Xtr = X_mlp_B.iloc[ti].copy(); ytr = y.iloc[ti]
    Xva = X_mlp_B.iloc[vi].copy(); yva = y.iloc[vi]
    Xte = X_test_mlp_B.copy()
    Xtr = pd.concat([Xtr, X_extra_mlp_B], ignore_index=True)
    ytr = pd.concat([ytr, y_extra],        ignore_index=True)
    te      = TargetEncoder(cv=FOLDS, smooth='auto', shuffle=True, random_state=SEED)
    te_enc  = te.fit_transform(Xtr[COMBO_NAMES_B], ytr)
    te_names= [f'_te_{c}' for c in COMBO_NAMES_B]
    Xtr[te_names] = te_enc
    Xva[te_names] = te.transform(Xva[COMBO_NAMES_B])
    Xte[te_names] = te.transform(Xte[COMBO_NAMES_B])
    m = RealMLP_TD_Classifier(**MLP_PARAMS)
    m.fit(Xtr, ytr, Xva, yva)
    oof_mlp_B[vi]  = m.predict_proba(Xva)[:, 1]
    pred_mlp_B    += m.predict_proba(Xte)[:, 1] / FOLDS
    print(f"{Fore.GREEN}{Style.BRIGHT}Fold {fold} AUC: {roc_auc_score(yva, oof_mlp_B[vi]):.5f}{Style.RESET_ALL}")
    torch.cuda.empty_cache(); gc.collect()

mlp_B_auc = roc_auc_score(y, oof_mlp_B)
print(f"\n{Fore.GREEN}{Style.BRIGHT}RealMLP-B OOF: {mlp_B_auc:.5f}{Style.RESET_ALL}")


## Model 6 — RealMLP-C: Wider architecture ~10 min
Hidden [1024,512,256,128], p_drop=0.10 — genuinely different minimum from A/B.

In [ ]:
%%time
MLP_PARAMS_C = {**MLP_PARAMS,
    'hidden_sizes': [1024, 512, 256, 128],
    'n_ens': 8,
    'p_drop': 0.10,
    'random_state': 123,
}

oof_mlp_C  = np.zeros(len(X))
pred_mlp_C = np.zeros(len(X_test))

for fold, (ti, vi) in enumerate(skf.split(X, y), 1):
    print(f"### Fold {fold}/{FOLDS}")
    Xtr = X_mlp_A.iloc[ti].copy(); ytr = y.iloc[ti]
    Xva = X_mlp_A.iloc[vi].copy(); yva = y.iloc[vi]
    Xte = X_test_mlp_A.copy()
    Xtr = pd.concat([Xtr, X_extra_mlp_A], ignore_index=True)
    ytr = pd.concat([ytr, y_extra],        ignore_index=True)
    te      = TargetEncoder(cv=FOLDS, smooth='auto', shuffle=True, random_state=SEED)
    te_enc  = te.fit_transform(Xtr[COMBO_NAMES_A], ytr)
    te_names= [f'_te_{c}' for c in COMBO_NAMES_A]
    Xtr[te_names] = te_enc
    Xva[te_names] = te.transform(Xva[COMBO_NAMES_A])
    Xte[te_names] = te.transform(Xte[COMBO_NAMES_A])
    m = RealMLP_TD_Classifier(**MLP_PARAMS_C)
    m.fit(Xtr, ytr, Xva, yva)
    oof_mlp_C[vi]  = m.predict_proba(Xva)[:, 1]
    pred_mlp_C    += m.predict_proba(Xte)[:, 1] / FOLDS
    print(f"{Fore.GREEN}{Style.BRIGHT}Fold {fold} AUC: {roc_auc_score(yva, oof_mlp_C[vi]):.5f}{Style.RESET_ALL}")
    torch.cuda.empty_cache(); gc.collect()

mlp_C_auc = roc_auc_score(y, oof_mlp_C)
print(f"\n{Fore.GREEN}{Style.BRIGHT}RealMLP-C OOF: {mlp_C_auc:.5f}{Style.RESET_ALL}")


In [ ]:
_names = ['LGB','XGB','GRU','RealMLP-A','RealMLP-B','RealMLP-C']
_oofs  = [oof_lgb, oof_xgb, oof_gru, oof_mlp_A, oof_mlp_B, oof_mlp_C]
_preds = [pred_lgb, pred_xgb, pred_gru, pred_mlp_A, pred_mlp_B, pred_mlp_C]

print("Individual OOF AUCs:")
for n, o in zip(_names, _oofs):
    print(f"  {n:14s}: {roc_auc_score(y, o):.5f}")


## Stacking Meta-Learner
Light LGB on 6 OOF predictions + raw race features, 5-fold CV.

In [ ]:
%%time
_raw_cols = ['TyreLife','RaceProgress','lap_in_stint','tyre_life_pct','laps_until_stop',
             'overdue_pit','drv_hist_overdue','in_ckt_pit_window','must_pit_signal',
             'urgency_composite','drv_champ_pos','driver_vs_avg_life','stint_degradation_ratio',
             'ckt_pit_avg_erg','laps_vs_ckt_erg','in_ckt_window_erg',
             'drv_hist_overdue_erg','ckt_deg_rate','expected_lt_loss',
             'pit_urgency_erg','tyre_degrading_faster','past_ckt_window',
             'ckt_drv_urgency','deg_rate_x_life',
             'avg_field_tyre_age','max_field_tyre_age','field_tyre_age_pct',
             'is_oldest_tyre','cars_older_tyres','tyre_age_vs_field',
             'n_pitted_this_lap','field_pit_rate','n_diff_compounds',
             'is_sc_proxy','laps_since_sc','in_sc_window',
             # v12 gap features
             'gap_to_car_ahead','gap_to_car_behind','in_drs_range',
             'undercut_viable','threat_from_behind','gap_ahead_delta',
             'car_ahead_pitted_now','car_behind_pitted_now','nearby_pit_pressure',
             'mandatory_pit_pending','mandatory_urgency','n_compounds_used',
             'is_vsc_proxy','is_full_sc_proxy','in_vsc_window','in_full_sc_window',
             'fuel_corrected_deg']
_avail = [c for c in _raw_cols if c in train_A.columns]
extra_tr = train_A[_avail].values.astype(np.float32)
extra_te = test_A[_avail].values.astype(np.float32)

stack_tr = np.column_stack(_oofs + [extra_tr])
stack_te = np.column_stack(_preds + [extra_te])

oof_stack  = np.zeros(len(y))
pred_stack = np.zeros(len(X_test))

meta_params = dict(
    objective='binary', metric='auc', n_estimators=500,
    learning_rate=0.03, num_leaves=15, min_child_samples=50,
    feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1,
    lambda_l1=2.0, lambda_l2=3.0,
    device=LGB_DEVICE, random_state=SEED, n_jobs=-1, verbose=-1,
)

for fold, (ti, vi) in enumerate(skf.split(stack_tr, y)):
    m = lgb.LGBMClassifier(**meta_params)
    m.fit(stack_tr[ti], y.iloc[ti],
          eval_set=[(stack_tr[vi], y.iloc[vi])],
          callbacks=[lgb.early_stopping(50, verbose=False)])
    oof_stack[vi]  = m.predict_proba(stack_tr[vi])[:, 1]
    pred_stack    += m.predict_proba(stack_te)[:, 1] / FOLDS
    print(f"  Fold {fold+1}: {roc_auc_score(y.iloc[vi], oof_stack[vi]):.5f}")

stack_auc = roc_auc_score(y, oof_stack)
print(f"\n{Fore.GREEN}{Style.BRIGHT}Stack OOF: {stack_auc:.5f}{Style.RESET_ALL}")


## v7 Internal Ensemble — Hill Climbing + Stacking

In [ ]:
%%time

def neg_auc(w):
    ww = np.abs(w); ww /= (ww.sum() + 1e-12)
    return -roc_auc_score(y, sum(wi * p for wi, p in zip(ww, _oofs)))

w0s = [
    [0.22, 0.10, 0.15, 0.18, 0.20, 0.15],
    [0.20, 0.08, 0.18, 0.15, 0.25, 0.14],
    [0.18, 0.08, 0.20, 0.18, 0.22, 0.14],
    [0.25, 0.08, 0.15, 0.15, 0.25, 0.12],
    [0.20, 0.06, 0.18, 0.22, 0.20, 0.14],
    [0.22, 0.10, 0.12, 0.18, 0.22, 0.16],
]

best_hc = None
for w0 in w0s:
    r = minimize(neg_auc, x0=w0, method='Nelder-Mead',
                 options={'maxiter': 3000, 'xatol': 1e-7, 'fatol': 1e-7})
    if best_hc is None or r.fun < best_hc.fun:
        best_hc = r

bw = np.abs(best_hc.x); bw /= bw.sum()
prob_oof  = sum(w * p for w, p in zip(bw, _oofs))
prob_pred = sum(w * p for w, p in zip(bw, _preds))
prob_auc  = roc_auc_score(y, prob_oof)

rank_oof  = np.average([rankdata(p)/len(p) for p in _oofs],  axis=0, weights=bw)
rank_pred = np.average([rankdata(p)/len(p) for p in _preds], axis=0, weights=bw)
rank_auc  = roc_auc_score(y, rank_oof)

eq_oof  = np.mean([rankdata(p)/len(p) for p in _oofs],  axis=0)
eq_pred = np.mean([rankdata(p)/len(p) for p in _preds], axis=0)
eq_auc  = roc_auc_score(y, eq_oof)

hc_auc  = max(prob_auc, rank_auc, eq_auc)
_hc_idx = [prob_auc, rank_auc, eq_auc].index(hc_auc)
hc_oof  = [prob_oof,  rank_oof,  eq_oof ][_hc_idx]
hc_pred = [prob_pred, rank_pred, eq_pred][_hc_idx]
hc_name = ['prob','rank','equal'][_hc_idx]

blend50_oof  = 0.5 * hc_oof  + 0.5 * oof_stack
blend50_pred = 0.5 * hc_pred + 0.5 * pred_stack
blend50_auc  = roc_auc_score(y, blend50_oof)

v7_cands = [
    (prob_auc,    prob_pred,    "hill-climb prob"),
    (rank_auc,    rank_pred,    "hill-climb rank"),
    (eq_auc,      eq_pred,      "equal rank"),
    (stack_auc,   pred_stack,   "stacking"),
    (blend50_auc, blend50_pred, "50/50 HC+Stack"),
]
v7_best_auc, v7_best_pred, v7_best_name = max(v7_cands, key=lambda x: x[0])
print(f"v12 internal best: {v7_best_name}  OOF={v7_best_auc:.5f}")
print(f"Weights: { {n: round(float(w), 3) for n, w in zip(_names, bw)} }")


In [ ]:
test_fe_ids = test_A['id'].values if 'id' in test_A.columns else TEST_IDS
pred_df  = pd.DataFrame({'id': test_fe_ids,
                         'PitNextLap': np.clip(v7_best_pred, 0.001, 0.999)})
sub_out  = sub[['id']].merge(pred_df, on='id', how='left')

assert sub_out.shape == (188165, 2),            f"Shape error: {sub_out.shape}"
assert sub_out['PitNextLap'].isna().sum() == 0, "NaN after merge!"
assert list(sub_out['id']) == list(sub['id']),  "ID order mismatch!"

sub_out.to_csv(f"{OUT}submission_v12_solo.csv", index=False)
print(f"submission_v12_solo.csv saved  (v12 standalone, OOF={v7_best_auc:.5f})")

raw_aucs = [lgb_auc, xgb_auc, gru_auc, mlp_A_auc, mlp_B_auc, mlp_C_auc, stack_auc]
summary  = pd.DataFrame({'Model':   _names + ['Stack','v12-solo'],
                         'OOF AUC': raw_aucs + [v7_best_auc]})
print(summary.to_string(index=False))

fi = (pd.DataFrame({'feature': TREE_FEATS,
                    'importance': lgb_models[0].feature_importances_})
      .sort_values('importance', ascending=False).head(25))
plt.figure(figsize=(10, 7))
plt.barh(fi['feature'][::-1], fi['importance'][::-1], color='steelblue')
plt.title('Top 25 LGB Feature Importances (v12)')
plt.tight_layout()
plt.savefig(f"{OUT}feature_importance_v12.png", dpi=120)
plt.show()


## 4-Way Blend: v12 + 3 Diverse Public Notebooks
v12 (in-memory) · ps6e5-ensemble (LB 0.9531) · realmlp-pytabkit · pit-or-stay.
Fixed DE blend collapse (concentration penalty 0.35×50, floor 0.10×30, equal-blend fallback). `submission.csv` will be the best blended output.

In [ ]:
N = len(sub)

def load_sub(path, label):
    df     = pd.read_csv(path)
    merged = sub[['id']].merge(df[['id','PitNextLap']], on='id', how='left')
    nans   = merged['PitNextLap'].isna().sum()
    if nans > 0:
        print(f"  ⚠️  {label}: {nans} NaN — filling 0.2")
        merged['PitNextLap'] = merged['PitNextLap'].fillna(0.2)
    p = merged['PitNextLap'].values.astype(np.float64)
    print(f"  {label:40s} | mean={p.mean():.4f} | std={p.std():.4f} | "
          f"min={p.min():.4f} | max={p.max():.4f}")
    return p

print("Loading public submissions...")
pub_subs = {}
for root in NB_ROOTS:
    matches = glob.glob(f"{root}/**/*.csv", recursive=True)
    sub_matches = [m for m in matches if 'submission' in os.path.basename(m).lower()]
    if not sub_matches:
        for m in matches:
            try:
                df = pd.read_csv(m, nrows=3)
                if 'id' in df.columns and 'PitNextLap' in df.columns:
                    sub_matches.append(m); break
            except: pass
    if sub_matches:
        nm = root.split('/')[-1][:35]
        p_candidate = load_sub(sorted(sub_matches)[-1], nm)
        # Skip miscalibrated submissions
        if p_candidate.max() < 0.85:
            print(f"  ⚠️  WARNING: {nm} — max={p_candidate.max():.3f} < 0.85, SKIPPING (miscalibrated)")
        # Skip near-duplicate submissions (keep first occurrence)
        elif any(np.corrcoef(rankdata(p_candidate), rankdata(existing))[0,1] > 0.9999
                 for existing in pub_subs.values()):
            print(f"  ⚠️  WARNING: {nm} — Spearman corr > 0.9999 with existing submission, SKIPPING (duplicate)")
        else:
            pub_subs[nm] = p_candidate
            print(f"  ✅ {nm}")
    else:
        print(f"  ❌ {root.split('/')[-1][:35]}: not found")

# v7 prediction is already in submission order (sub_out was verified above)
your_v12_pred = sub_out['PitNextLap'].values.astype(np.float64)
print(f"\nv12 (in-memory)                          | "
      f"mean={your_v12_pred.mean():.4f} | std={your_v12_pred.std():.4f} | "
      f"min={your_v12_pred.min():.4f} | max={your_v12_pred.max():.4f}")

blend_subs  = {'your_v12': your_v12_pred, **pub_subs}
blend_names = list(blend_subs.keys())
blend_preds = [blend_subs[n] for n in blend_names]

print(f"\nTotal in blend: {len(blend_names)} | Order: {blend_names}")
if len(blend_names) < 3:
    print(f"⚠️  Only {len(blend_names)} sources — attach public notebooks")
elif len(blend_names) == 4:
    print(f"✅ Full 4-way blend: v9 + ps6e5-ensemble + realmlp-pytabkit + pit-or-stay")


In [ ]:
ranks_list  = [rankdata(p) / N for p in blend_preds]
corr_matrix = np.corrcoef(ranks_list)

print("Spearman rank correlation (lower = more diverse):")
df_corr = pd.DataFrame(corr_matrix,
                        index=[n[:25] for n in blend_names],
                        columns=[n[:25] for n in blend_names])
print(df_corr.round(4).to_string())

triu = np.triu_indices(len(blend_names), 1)
for i, j in zip(*triu):
    c = corr_matrix[i, j]
    tag = "⚠️  IDENTICAL — remove one!" if c > 0.9999 else ("✅ DIVERSE" if c < 0.97 else "")
    print(f"  {blend_names[i][:22]:22s} ↔ {blend_names[j][:22]:22s}  {c:.4f}  {tag}")


## Weight Combinations
Order: `[your_v12, ps6e5-ensemble, realmlp-pytabkit, pit-or-stay]`
v12 anchors the blend; 3 diverse public notebooks add coverage.

In [ ]:
n_blend = len(blend_names)
print(f"Blend sources ({n_blend}): {blend_names}")

# Build weight combos dynamically — always exactly n_blend elements
def make_combo(weights_dict, label):
    w = np.array([weights_dict.get(n, 0.0) for n in blend_names])
    w = np.maximum(w, 0)
    if w.sum() < 1e-9:
        w = np.ones(n_blend) / n_blend
    return w / w.sum(), label

SPECIFIC_COMBOS = [
    make_combo({'your_v12': 0.35, 'ps6e5-ensemble-0-95314-best-score': 0.25,
                'ps-s6-e5-realmlp-pytabkit': 0.25,
                'pit-or-stay-f1-strategy-1': 0.15}, "v12_heavy"),
    make_combo({'your_v12': 0.30, 'ps6e5-ensemble-0-95314-best-score': 0.25,
                'ps-s6-e5-realmlp-pytabkit': 0.25,
                'pit-or-stay-f1-strategy-1': 0.20}, "balanced"),
    make_combo({'your_v12': 0.40, 'ps-s6-e5-realmlp-pytabkit': 0.35,
                'pit-or-stay-f1-strategy-1': 0.25}, "v12_realmlp_pitstay"),
    make_combo({'your_v12': 0.50, 'ps6e5-ensemble-0-95314-best-score': 0.30,
                'pit-or-stay-f1-strategy-1': 0.20}, "v12_dominant"),
]

combo_blends = []
for w, label in SPECIFIC_COMBOS:
    blend = np.average(ranks_list, axis=0, weights=w)
    combo_blends.append((w, label, blend))
    print(f"  {label:35s} std={blend.std():.5f}  weights={w.round(3)}")

for i, (w, label, b) in enumerate(combo_blends, 1):
    out_df = sub[['id']].copy()
    out_df['PitNextLap'] = np.clip(b, 0.001, 0.999)
    assert out_df.shape == (188165, 2)
    out_df.to_csv(f"{OUT}submission_blend_{i}_{label[:20]}.csv", index=False)
    print(f"  Saved submission_blend_{i}_{label[:20]}.csv")


## Improved Blend Methods
DE-anchored · Nelder-Mead anchored · Geometric mean · Bayesian

In [ ]:
from scipy.optimize import differential_evolution
import time as _time

n_blend  = len(blend_names)
EPS      = 1e-9

your_idx = blend_names.index('your_v12') if 'your_v12' in blend_names else 0

KNOWN_LB = {
    'your_v12':                              float(v7_best_auc),
    'ps6e5-ensemble-0-95314-best-score':     0.9531,
    'ps-s6-e5-realmlp-pytabkit':            0.9510,
    'pit-or-stay-f1-strategy-1':            0.9510,
}

# Precompute centred/normalised v7 rank vector — avoids repeated rankdata calls
# Since blend is a weighted average of rank arrays, Pearson ≡ Spearman here.
_v7  = ranks_list[your_idx] - ranks_list[your_idx].mean()
_v7n = _v7 / (np.linalg.norm(_v7) + EPS)

# Stack ranks into matrix for fast np.dot instead of per-call np.average
_RM  = np.stack(ranks_list, axis=0)   # shape: (n_blend, N)

def neg_oof_anchored(log_w):
    w = np.exp(log_w); w /= w.sum()
    blend = w @ _RM                          # (N,) — fast matrix multiply
    b_c   = blend - blend.mean()
    corr  = np.dot(b_c, _v7n) / (np.linalg.norm(b_c) + EPS)
    div   = 1.0 - corr
    anchor_penalty = max(0.0, 0.25 - w[your_idx]) * 20.0
    conc_penalty   = max(0.0, w.max() - 0.35) * 50.0
    floor_penalty  = sum(max(0.0, 0.10 - wi) * 30.0 for wi in w)
    return -(float(v7_best_auc) + 0.008 * div) + anchor_penalty + conc_penalty + floor_penalty

log_w0 = np.log(np.ones(n_blend) / n_blend)
bounds  = [(log_w0[i] - 2.5, log_w0[i] + 2.5) for i in range(n_blend)]

_t0 = _time.time()
de_result = differential_evolution(
    neg_oof_anchored, bounds, seed=SEED,
    maxiter=80, tol=1e-4, polish=False,
    popsize=8, mutation=(0.5, 1.2), recombination=0.7,
)
w_de = np.exp(de_result.x); w_de /= w_de.sum()
if w_de.max() > 0.40:
    print(f"⚠️  DE collapsed (max_weight={w_de.max():.3f}) — using equal blend")
    w_de = np.ones(n_blend) / n_blend
blend_de = w_de @ _RM
print(f"DE done in {_time.time()-_t0:.1f}s | weights: {dict(zip([n[:18] for n in blend_names], w_de.round(3)))}")
print(f"DE blend std: {blend_de.std():.5f}")

best_nm = None
for w0_raw in [
    np.ones(n_blend) / n_blend,
    np.array([0.40] + [0.60/(n_blend-1)] * (n_blend-1)),
    np.array([0.35, 0.25] + [0.40/max(1, n_blend-2)] * max(1, n_blend-2))[:n_blend],
]:
    w0_raw = w0_raw[:n_blend]; w0_raw /= w0_raw.sum()
    r = minimize(neg_oof_anchored, x0=np.log(w0_raw + EPS),
                 method='Nelder-Mead',
                 options={'maxiter': 800, 'xatol': 1e-5, 'fatol': 1e-5})
    if best_nm is None or r.fun < best_nm.fun:
        best_nm = r

w_nm     = np.exp(best_nm.x); w_nm /= w_nm.sum()
blend_nm = w_nm @ _RM
print(f"NM weights: {dict(zip([n[:18] for n in blend_names], w_nm.round(3)))}")

blend_geo = np.exp(np.mean([np.log(r + EPS) for r in ranks_list], axis=0))
blend_geo = rankdata(blend_geo) / N

blend_eq = np.mean(ranks_list, axis=0)

scores_arr = np.array([KNOWN_LB.get(n, 0.951) for n in blend_names])
try:
    R_inv = np.linalg.inv(corr_matrix + 1e-5 * np.eye(n_blend))
    w_bay = np.maximum(R_inv @ scores_arr, 0)
    if w_bay.sum() > 1e-9:
        w_bay /= w_bay.sum()
    else:
        w_bay = np.ones(n_blend) / n_blend
    blend_bay = w_bay @ _RM
except:
    blend_bay = blend_eq

all_blends = {
    'de_anchored':    blend_de,
    'nm_anchored':    blend_nm,
    'geometric_mean': blend_geo,
    'equal_rank':     blend_eq,
    'bayesian':       blend_bay,
}
for w, label, b in combo_blends:
    all_blends[f'combo_{label[:20]}'] = b

print(f"\n{'Method':38s}  std        div_from_v9")
for bname, b in all_blends.items():
    b_c  = b - b.mean()
    corr = np.dot(b_c, _v7n) / (np.linalg.norm(b_c) + EPS)
    print(f"  {bname:36s}  {b.std():.5f}    {1-corr:.5f}")

PRIMARY_BLEND      = blend_de
PRIMARY_BLEND_NAME = 'de_anchored'
print(f"\n→ Primary blend: {PRIMARY_BLEND_NAME}")


In [ ]:
for bname, b in all_blends.items():
    out_df = sub[['id']].copy()
    out_df['PitNextLap'] = np.clip(b, 0.001, 0.999)
    out_df.to_csv(f"{OUT}submission_{bname[:35]}.csv", index=False)

main_out = sub[['id']].copy()
main_out['PitNextLap'] = np.clip(PRIMARY_BLEND, 0.001, 0.999)
assert main_out.shape == (188165, 2)
assert main_out['PitNextLap'].isna().sum() == 0
main_out.to_csv(f"{OUT}submission.csv", index=False)
print(f"✅ submission.csv saved = {PRIMARY_BLEND_NAME}")
print(f"Expected LB: {v7_best_auc:.5f} (solo) → {v7_best_auc + 0.002:.5f}+ (blended)")


In [ ]:
print("="*70)
print("FINAL REPORT")
print("="*70)
print(f"\nv12 internal models:")
for n, a in zip(_names + ['Stack'], [lgb_auc, xgb_auc, gru_auc, mlp_A_auc, mlp_B_auc, mlp_C_auc, stack_auc]):
    print(f"  {n:14s}: {a:.5f}")
print(f"  v7 best      : {v7_best_auc:.5f}  ({v7_best_name})")

print(f"\nBlend participants ({len(blend_names)}):")
for n in blend_names:
    print(f"  - {n}")

print(f"\nPairwise diversity:")
for i, j in zip(*triu):
    div = "★ DIVERSE" if corr_matrix[i,j] < 0.97 else ""
    print(f"  {blend_names[i][:22]:22s} ↔ {blend_names[j][:22]:22s}  {corr_matrix[i,j]:.4f}  {div}")

print(f"\nFiles saved:")
print(f"  submission.csv           — PRIMARY BLEND ({PRIMARY_BLEND_NAME})")
print(f"  submission_v12_solo.csv   — v12 standalone (OOF={v7_best_auc:.5f})")
for bname in all_blends:
    print(f"  submission_{bname[:35]}.csv")

print(f"\n{Fore.CYAN}{Style.BRIGHT}⚡ Primary submission: submission.csv ({PRIMARY_BLEND_NAME})")
print(f"   v7 OOF AUC: {v7_best_auc:.5f}")
print(f"   Expected LB: ~{v7_best_auc:.5f} (solo) → {v7_best_auc+0.002:.5f}+ (blended){Style.RESET_ALL}")
print(f"\nTip: if public notebooks are not attached, submission_v12_solo.csv is still valid.")
